## Flow Accumulation from Google Earth Engine

This notebook fetches **flow accumulation** (upstream drainage area in number of cells) for points given latitude and longitude, using the [Google Earth Engine (GEE) Python API](https://developers.google.com/earth-engine/guides/python_install) and the [WWF HydroSHEDS Flow Accumulation](https://developers.google.com/earth-engine/datasets/catalog/WWF_HydroSHEDS_15ACC) dataset.

### Prerequisites

1. **Install** the Earth Engine API: `pip install earthengine-api`
2. **Authenticate** (one-time per machine): run the cell with `ee.Authenticate()`. This opens a browser to sign in with your Google account and grant Earth Engine access.
3. **Initialize** with a Google Cloud project you have access to (or leave default): `ee.Initialize(project='your-project')`

### Dataset

- **WWF/HydroSHEDS/15ACC** – 15 arc-second (~464 m) resolution. Values are the number of upstream cells draining into each cell (1 at river sources to millions at river mouths). Quality is lower above 60°N (no SRTM there).

In [8]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os

import ee

### 1. Authenticate and initialize (run once)

Run the next cell once to authenticate. If you already have credentials, you can skip to the initialize cell.

In [9]:
# One-time: opens browser to sign in with Google and grant Earth Engine access
# ee.Authenticate()

In [10]:
# Initialize with default project, or set project='your-gcp-project'
try:
    ee.Initialize()
    print("Earth Engine initialized.")
except Exception as e:
    print("Run ee.Authenticate() first, then ee.Initialize(). Error:", e)

Earth Engine initialized.


### 2. Load points (Latitude, Longitude)

Use a CSV with `Latitude` and `Longitude`. Path is relative to the project root or set `DATA_DIR`.

In [11]:
# Paths: relative to project root; if running from topography_features/, use parent dir
if os.path.exists("data"):
    DATA_DIR = "data"
else:
    DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
ORIGINAL_DIR = os.path.join(DATA_DIR, "original")

# Load locations: use elevation_gradient_locations or any CSV with Latitude, Longitude
# locations_path = os.path.join(PROCESSED_DIR, "elevation_gradient_locations.csv")
# if not os.path.exists(locations_path):
#     locations_path = os.path.join(ORIGINAL_DIR, "submission_template.csv")

locations_path = os.path.join(ORIGINAL_DIR, "submission_template.csv")

df = pd.read_csv(locations_path)
print(locations_path)
locations = df[["Latitude", "Longitude"]].drop_duplicates().reset_index(drop=True)
print(f"Unique locations: {len(locations)}")

/Users/ben/Documents/Projects/ey_challenge_2026/data/original/submission_template.csv
Unique locations: 24


### 3. Build GEE FeatureCollection of points

Earth Engine uses (longitude, latitude) for coordinates. We add an `id` property so we can match results back to the dataframe.

In [12]:
def points_to_feature_collection(locations_df):
    """Build ee.FeatureCollection from DataFrame with Latitude, Longitude."""
    features = []
    for i, row in locations_df.iterrows():
        lon, lat = float(row["Longitude"]), float(row["Latitude"])
        pt = ee.Geometry.Point([lon, lat])
        feat = ee.Feature(pt, {"id": i})
        features.append(feat)
    return ee.FeatureCollection(features)

points_fc = points_to_feature_collection(locations)
print(f"FeatureCollection with {locations.shape[0]} points.")

FeatureCollection with 24 points.


### 4. Load HydroSHEDS flow accumulation and sample at points

We use **WWF/HydroSHEDS/15ACC** (15 arc-second, ~464 m). Band `b1` is flow accumulation (number of upstream cells).

In [13]:
# HydroSHEDS 15 arc-second flow accumulation; scale ~464 m
HYDROSHEDS_ACC = "WWF/HydroSHEDS/15ACC"
SCALE_M = 463  # meters (15 arc-seconds at equator)

flow_acc_image = ee.Image(HYDROSHEDS_ACC).select("b1")

# Sample image at each point (one value per point)
sampled = flow_acc_image.sampleRegions(
    collection=points_fc,
    scale=SCALE_M,
    geometries=False,
)

# Transfer results to client (for small point sets; for large sets consider Export)
result = sampled.getInfo()

### 5. Parse results and merge with locations

Extract flow accumulation (band `b1`) and sort by `id` to align with the locations dataframe.

In [14]:
def parse_flow_accumulation_result(result):
    """Parse getInfo() result into list of (id, flow_accumulation)."""
    features = result.get("features", [])
    rows = []
    for f in features:
        props = f.get("properties", {})
        idx = props.get("id")
        # b1 is the band name in HydroSHEDS 15ACC
        acc = props.get("b1")
        if acc is None:
            acc = np.nan
        rows.append({"id": idx, "flow_accumulation": acc})
    return pd.DataFrame(rows).sort_values("id").reset_index(drop=True)

acc_df = parse_flow_accumulation_result(result)
locations["flow_accumulation"] = acc_df["flow_accumulation"].values
locations.head(10)

,Latitude,Longitude,flow_accumulation
0,-32.043333,27.822778,2
1,-33.329167,26.077500,8091
2,-32.991639,27.640028,6087
3,-34.096389,24.439167,834
4,-32.000556,28.581667,30514
5,-32.086390,25.575560,60005
6,-33.185361,27.390750,14097
7,-33.731111,24.618333,163529
8,-31.905000,25.430000,9406
9,-32.515278,28.015556,103429


### 6. Save and optionally merge with full dataset

Save unique locations with flow accumulation. Optionally merge back into a full dataframe (e.g. with elevation/gradient) on `Latitude` and `Longitude`.

In [15]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
out_path = os.path.join(PROCESSED_DIR, "submission_flow_accumulation_locations.csv")
locations.to_csv(out_path, index=False)
print(f"Saved {len(locations)} rows to {out_path}")

Saved 24 rows to /Users/ben/Documents/Projects/ey_challenge_2026/data/processed/submission_flow_accumulation_locations.csv


In [16]:
# Optional: merge flow accumulation into the original dataframe (e.g. elevation_gradient_locations)
df_merged = df.merge(
    locations[["Latitude", "Longitude", "flow_accumulation"]],
    on=["Latitude", "Longitude"],
    how="left",
)
df_merged.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,flow_accumulation
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN,2
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,8091
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN,6087
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,834
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN,30514


### Notes

- **Scale**: 15 arc-second (~464 m). For finer resolution, GEE also has [HydroSHEDS 3ACC](https://developers.google.com/earth-engine/datasets/catalog/WWF_HydroSHEDS_3ACC) (~90 m); use a smaller `scale` when sampling.
- **Large point sets**: `getInfo()` can be slow or hit size limits. For thousands of points, export the sampled FeatureCollection to Google Drive or Cloud Storage with `ee.batch.Export.table.toDrive()` and load the result in a separate step.
- **Interpretation**: Flow accumulation is in *number of cells*; cell size varies with latitude, so it is not directly in km². Use it as a relative measure of upstream drainage (e.g. for water quality or runoff context).